In [1]:
# ====================================================
# NEAR MISS CLEANING (FINAL)
# OCT - DEC ONLY
# ====================================================

import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings("ignore")

# ====================================================
# LOAD
# ====================================================

file = "../1.Near Misses Report database 2010.xlsx"

xls = pd.ExcelFile(file)

print("Sheets:")
print(xls.sheet_names)

sheet = next(
    s for s in xls.sheet_names
    if "oct" in s.lower() and "dec" in s.lower()
)

df = pd.read_excel(
    file,
    sheet_name=sheet
)

original_shape = df.shape

print(f"\nLoaded Sheet: {sheet}")
print(f"Original Shape: {original_shape}")

# ====================================================
# MERGE POSSIBLE CONSEQUENCE
# ====================================================

base = "Possible Consequence (Can be more than one)"

unnamed = [
    c
    for c in df.columns
    if str(c).startswith("Unnamed:")
]

if base in df.columns:

    merge_cols = [base] + unnamed

    df[base] = (
        df[merge_cols]
        .fillna("")
        .astype(str)
        .apply(
            lambda x:
            "\n".join(
                [
                    v.strip()
                    for v in x
                    if v.strip()
                    and v != "nan"
                ]
            ),
            axis=1
        )
    )

    df.drop(
        columns=unnamed,
        inplace=True
    )

# ====================================================
# CLEAN COLUMN NAMES
# ====================================================

def clean(col):

    col = str(col).strip().lower()

    col = re.sub(
        r"[^\w]+",
        "_",
        col
    )

    col = re.sub(
        "_+",
        "_",
        col
    )

    return col.strip("_")


df.columns = [
    clean(c)
    for c in df.columns
]

# ====================================================
# CLEAN TEXT
# ====================================================

text_cols = df.select_dtypes(
    include="object"
).columns

for col in text_cols:

    df[col] = (
        df[col]
        .astype("string")
        .replace(
            r"^\s*$",
            np.nan,
            regex=True
        )
        .str.strip()
    )

# ====================================================
# FIX DATE FORMAT
# ====================================================

date_cols=[

    c

    for c in df.columns

    if (
        "date" in c
        and
        "update" not in c
    )
]

def format_date(x):

    if pd.isna(x):
        return np.nan

    x=str(x).strip()

    if x.lower() in [
        "na",
        "n/a",
        "nan",
        "not mentioned"
    ]:
        return np.nan

    # already correct → keep
    if re.match(
        r"^\d{2}-[A-Za-z]{3}-\d{2}$",
        x
    ):
        return x

    try:

        dt=pd.to_datetime(
            x,
            errors="coerce"
        )

        if pd.notna(dt):

            return dt.strftime(
                "%d-%b-%y"
            )

    except:
        pass

    return np.nan


for col in date_cols:

    df[col] = df[col].apply(
        format_date
    )

# ====================================================
# REMOVE EMPTY COLUMNS
# ====================================================

drop_cols = []

for c in [
    "master",
    "chief_engineer",
    "superintendent"
]:

    if (
        c in df.columns
        and
        df[c].isna().all()
    ):

        drop_cols.append(c)

df.drop(
    columns=drop_cols,
    inplace=True
)

# ====================================================
# REMOVE WEAK INCIDENT ROWS
# ====================================================

important = [

    "description_of_the_incident_and_cause_analysis",

    "incident_category_potential",

    "location_of_the_unsafe_act_condition",

    "possible_consequence_can_be_more_than_one"

]

important = [

    c
    for c in important
    if c in df.columns
]

before = len(df)

df = df.dropna(
    subset=important,
    thresh=2
)

removed = before - len(df)

# ====================================================
# DATE CORRECTIVE ACTION RULES
# ====================================================

action = "date_corrective_action_completed"
status = "status"
office = "suggested_corrective_action_office_use"

pending_words = [

    "forthcoming safety meeting",

    "discuss same",

    "please ensure",

    "kindly discuss",

    "discuss this incident",

    "safety meeting",

    "due date",

    "to be",

    "shall be",

    "proposed",

    "modify",

    "modified",

    "request",

    "requested",

    "review",

    "reviewed",

    "ensure",

    "continue to",

    "amendment",

    "to include"

]

resolved_words = [

    "resolved",

    "completed",

    "job completed",

    "implemented",

    "rectified",

    "closed",

    "confirmed"

]

if action in df.columns:

    df[action] = (
        df[action]
        .astype("string")
        .replace(
            ["NA", "N/A", ""],
            np.nan
        )
        .str.strip()
    )

if status in df.columns:

    df[status] = (
        df[status]
        .astype("string")
        .str.strip()
    )

if (
    action in df.columns
    and
    status in df.columns
):

    for i in df.index:

        act = str(
            df.at[i, action]
        ).lower()

        st = str(
            df.at[i, status]
        ).lower()

        office_text = ""

        if office in df.columns:

            office_text = str(
                df.at[i, office]
            ).lower()

        pending = any(
            w in office_text
            for w in pending_words
        )

        resolved = any(
            w in office_text
            for w in resolved_words
        )

        if (
            pd.isna(df.at[i, action])
            or act == "nan"
        ):

            if pending:

                df.at[i, action] = "Pending"
                df.at[i, status] = "Open"

            elif resolved:

                df.at[i, action] = "Resolved"
                df.at[i, status] = "Closed"

            elif st == "open":

                df.at[i, action] = "Pending"

            elif st == "closed":

                df.at[i, action] = "Resolved"

        else:

            act_text = str(
                df.at[i, action]
            ).lower()

            if "pending" in act_text:

                df.at[i, status] = "Open"

            elif any(
                w in act_text
                for w in resolved_words
            ):

                df.at[i, status] = "Closed"

# ====================================================
# REMOVE DUPLICATES
# ====================================================

dup = df.duplicated().sum()

df.drop_duplicates(
    inplace=True
)

# ====================================================
# MISSING SUMMARY
# ====================================================

missing = df.isna().sum()

# ====================================================
# EXPORT
# ====================================================

output = "cleaned_1_Near_Miss_2010.xlsx"

with pd.ExcelWriter(
    output,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        index=False
    )

    ws = writer.sheets["Sheet1"]

    for col in ws.columns:

        width = max(

            len(str(cell.value))
            if cell.value
            else 0

            for cell

            in col

        )

        ws.column_dimensions[
            col[0].column_letter
        ].width = min(
            width + 3,
            80
        )

# ====================================================
# REPORT
# ====================================================

print("\n========== SUMMARY ==========")

print(f"Original Shape: {original_shape}")
print(f"Final Shape: {df.shape}")

print(f"Rows Removed: {removed}")

print(f"Duplicates Removed: {dup}")

print("\nColumns Removed:")
print(drop_cols)

print("\nMissing Values Summary:")
print(
    missing[
        missing > 0
    ]
)

print(f"\nTotal Missing Cells: {missing.sum()}")

print(f"\nSaved: {output}")

Sheets:
['Near Miss - Jan - March 2010', 'April to June - 2010', 'July - Sep - 2010', 'Sheet1', 'Jul - Sep - 2010', 'Oct - Dec', 'List']

Loaded Sheet: Oct - Dec
Original Shape: (80, 22)

========== SUMMARY ==========
Original Shape: (80, 22)
Final Shape: (72, 16)
Rows Removed: 8
Duplicates Removed: 0

Columns Removed:
['master', 'chief_engineer', 'superintendent']

Missing Values Summary:
primary_contributing_factor    1
area_of_concern                3
dtype: int64

Total Missing Cells: 4

Saved: cleaned_1_Near_Miss_2010.xlsx
